# Лабораторная работа №1


Kaggle: https://www.kaggle.com/datasets/hubertsidorowicz/steam-games-dataset-daily-updates

В этом блокноте будем проводить очистку датасета и последующее сохранение в Iceberg.


## 1) Инициализация Spark


In [1]:
from src.spark_session import create_spark

spark = create_spark("Lab_1_Data_Cleaning")
spark

Picked up JAVA_TOOL_OPTIONS: -Djava.net.preferIPv4Stack=true
Picked up JAVA_TOOL_OPTIONS: -Djava.net.preferIPv4Stack=true


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /tmp/ivy/cache
The jars for the packages stored in: /tmp/ivy/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5961c2fe-cf38-4928-a8ac-a35d08124f26;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.11.0 in central
:: resolution report :: resolve 377ms :: artifacts dl 10ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.11.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-

## 2) Пути и имя таблицы


In [2]:
INPUT_PATH = "/app/data/raw/steam_games.csv"
TABLE_NAME = "local.lab1.steam_games"

# None — использовать весь датасет.
# Для тестового запуска можно указать, например, 100_000.
ROW_LIMIT = None

print("CSV:", INPUT_PATH)
print("Iceberg:", TABLE_NAME)
print("Limit:", ROW_LIMIT)


CSV: /app/data/raw/steam_games.csv
Iceberg: local.lab1.steam_games
Limit: None


## 3) Чтение исходного CSV

В текстовых полях встречаются переносы строк, поэтому используется `multiLine=true`. Кавычки и экранирование также задаются явно.


In [3]:
df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("mode", "PERMISSIVE")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .csv(INPUT_PATH)
)

if ROW_LIMIT is not None:
    df_raw = df_raw.limit(ROW_LIMIT)

# В исходном CSV у первого столбца с Steam App ID пустой заголовок.
first_column = df_raw.columns[0]
if first_column != "app_id":
    df_raw = df_raw.withColumnRenamed(first_column, "app_id")

print("Количество столбцов:", len(df_raw.columns))
print("Количество строк:", df_raw.count())
print("Количество партиций:", df_raw.rdd.getNumPartitions())
df_raw.printSchema()


Количество столбцов: 42


Количество строк: 140243
Количество партиций: 1
root
 |-- app_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- price: string (nullable = true)
 |-- price_status: string (nullable = true)
 |-- estimated_owners: string (nullable = true)
 |-- developers: string (nullable = true)
 |-- publishers: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- positive: string (nullable = true)
 |-- negative: string (nullable = true)
 |-- recommendations: string (nullable = true)
 |-- peak_ccu: string (nullable = true)
 |-- metacritic_score: string (nullable = true)
 |-- user_score: string (nullable = true)
 |-- average_playtime_forever: string (nullable = true)
 |-- median_playtime_forever: string (nullable = true)
 |-- average_playtime_2weeks: string (nullable = true)
 |-- median_playtime_2weeks: string (nullable = true)
 |-- short_description:

## 4) Проверка корректности чтения


In [4]:
from pyspark.sql.functions import col, sum as spark_sum, when

df_raw.select(
    "app_id",
    "name",
    "genres",
    "categories",
    "tags",
    "positive",
    "negative",
    "price",
).show(10, truncate=False)

# Быстрая проверка, что записи CSV не разъехались из-за переносов строк.
df_raw.select(
    spark_sum(when(col("app_id").isNull(), 1).otherwise(0)).alias("app_id_null"),
    spark_sum(when(col("name").isNull(), 1).otherwise(0)).alias("name_null"),
    spark_sum(when(col("release_date").isNull(), 1).otherwise(0)).alias("release_date_null"),
).show()


+------+--------------------+-------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------+--------+-----+
|app_id|name                |genres                                                       |categories                                                                                                                                                           |tags                                                                                                                                                                                                                     

+-----------+---------+-----------------+
|app_id_null|name_null|release_date_null|
+-----------+---------+-----------------+
|          0|       16|               83|
+-----------+---------+-----------------+



## 5) Выгрузка 1000 строк для ручного просмотра (не проводить)

In [5]:
from pathlib import Path

SAMPLE_OUTPUT_PATH = "/app/output/results/steam_sample_1000.csv"

sample_columns = [
    "app_id",
    "name",
    "release_date",
    "price",
    #"price_status",
    "estimated_owners", # разбить на 3 колонки (минимум, максимум, среднее)
    #"developers",
    #"publishers",
    "genres", # категории игр
    "categories",
    "tags",
    "positive", # позтивные и негативные комменты под играми (мб ввести отношение их)
    "negative",
    "recommendations",
    "peak_ccu",
    "metacritic_score",
    #"user_score",
    "average_playtime_forever",
    "median_playtime_forever",
    #"average_playtime_2weeks",
    #"median_playtime_2weeks",
    "achievements",
    "dlc_count",
    "supported_languages", # заменить на количество языков
    "full_audio_languages", # заменить на количество языков
    "windows",
    "mac",
    "linux",
    #"steam_store_available",
    #"steam_spy_available",
]

df_sample = df_raw.select(*sample_columns).limit(1500)

Path(SAMPLE_OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)
df_sample.toPandas().to_csv(
    SAMPLE_OUTPUT_PATH,
    index=False,
    encoding="utf-8",
)

print("CSV сохранён:", SAMPLE_OUTPUT_PATH)
print("Строк:", df_sample.count())
print("Столбцов:", len(df_sample.columns))


CSV сохранён: /app/output/results/steam_sample_1000.csv
Строк: 1500
Столбцов: 22


## 6) Выбор признаков

In [6]:
SELECTED_COLUMNS = [
    "app_id",
    "name",
    "release_date",
    "price",
    "estimated_owners", # разбить на 3 колонки (минимум, максимум, среднее)
    "genres", # категории игр
    "categories",
    "tags",
    "positive", # позтивные и негативные комменты под играми (мб ввести отношение их)
    "negative",
    "recommendations",
    "peak_ccu",
    "metacritic_score",
    "average_playtime_forever",
    "median_playtime_forever",
    "achievements",
    "dlc_count",
    "supported_languages", # заменить на количество языков
    "full_audio_languages", # заменить на количество языков
    "windows",
    "mac",
    "linux",
]

df = df_raw.select(*SELECTED_COLUMNS)

df.show(10, truncate=False)

+------+--------------------+------------+-----+----------------+-------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------+--------+---------------+--------+----------------+------------------------+-----------------------+------------+---------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## 7) Приведение типов и нормализация разных форматов признаков

В датасете встречаются разные представления одних и тех же признаков. Перед сохранением в Iceberg они приводятся к единой схеме.


In [7]:
from pyspark.sql.functions import (
    col,
    filter as array_filter,
    from_json,
    length,
    lit,
    map_keys,
    regexp_count,
    regexp_extract,
    regexp_replace,
    size,
    split,
    to_date,
    transform,
    trim,
    when,
)

from pyspark.sql.types import (
    ArrayType,
    LongType,
    MapType,
    StringType,
    StructField,
    StructType,
)


string_array_schema = ArrayType(StringType())

description_array_schema = ArrayType(
    StructType([
        StructField("description", StringType(), True),
    ])
)

tags_map_schema = MapType(
    StringType(),
    LongType(),
)


def normalize_description_array(column_name):
    """
    Приводит два формата к array<string>:
    ["Action", "Indie"]
    [{"id": "...", "description": "Action"}, ...]
    """
    raw = col(column_name)

    return when(
        raw.contains('"description"'),
        transform(
            from_json(raw, description_array_schema),
            lambda item: item["description"],
        ),
    ).otherwise(
        from_json(raw, string_array_schema)
    )


def normalize_tags(column_name):
    """
    Приводит два формата тегов к array<string>:
    ["Action", "Indie"]
    {"Action": 120, "Indie": 95}
    """
    raw = col(column_name)

    return when(
        trim(raw).startswith("{"),
        map_keys(
            from_json(raw, tags_map_schema)
        ),
    ).otherwise(
        from_json(raw, string_array_schema)
    )


df = (
    df

    # -------------------------------------------------
    # Основные признаки
    # -------------------------------------------------

    .withColumn(
        "app_id",
        col("app_id").cast("long"),
    )

    .withColumn(
        "release_date",
        to_date(
            col("release_date"),
            "yyyy-MM-dd",
        ),
    )

    .withColumn(
        "price",
        col("price").cast("double"),
    )

    # -------------------------------------------------
    # estimated_owners
    #
    # Поддерживаются оба варианта:
    # "50000 - 100000"
    # "50,000 .. 100,000"
    # -------------------------------------------------

    .withColumn(
        "_estimated_owners_clean",
        regexp_replace(
            col("estimated_owners"),
            ",",
            "",
        ),
    )

    .withColumn(
        "estimated_owners_min",
        regexp_extract(
            col("_estimated_owners_clean"),
            r"^\s*(\d+)",
            1,
        ).cast("long"),
    )

    .withColumn(
        "estimated_owners_max",
        regexp_extract(
            col("_estimated_owners_clean"),
            r"(\d+)\s*$",
            1,
        ).cast("long"),
    )

    .withColumn(
        "estimated_owners_avg",
        (
            col("estimated_owners_min")
            + col("estimated_owners_max")
        ) / 2,
    )

    # -------------------------------------------------
    # Множественные категориальные признаки
    #
    # genres/categories:
    #   ["Action", "Indie"]
    #   [{"id": "...", "description": "Action"}, ...]
    #
    # tags:
    #   ["Action", "Indie"]
    #   {"Action": 120, "Indie": 95}
    # -------------------------------------------------

    .withColumn(
        "genres",
        normalize_description_array("genres"),
    )

    .withColumn(
        "categories",
        normalize_description_array("categories"),
    )

    .withColumn(
        "tags",
        normalize_tags("tags"),
    )

    # -------------------------------------------------
    # Отзывы и активность
    # -------------------------------------------------

    .withColumn(
        "positive",
        col("positive").cast("long"),
    )

    .withColumn(
        "negative",
        col("negative").cast("long"),
    )

    .withColumn(
        "recommendations",
        col("recommendations")
        .cast("double")
        .cast("long"),
    )

    .withColumn(
        "peak_ccu",
        col("peak_ccu").cast("long"),
    )

    .withColumn(
        "metacritic_score",
        col("metacritic_score")
        .cast("double")
        .cast("integer"),
    )

    # -------------------------------------------------
    # Время игры
    # -------------------------------------------------

    .withColumn(
        "average_playtime_forever",
        col("average_playtime_forever").cast("long"),
    )

    .withColumn(
        "median_playtime_forever",
        col("median_playtime_forever").cast("long"),
    )

    # -------------------------------------------------
    # Достижения и DLC
    # -------------------------------------------------

    .withColumn(
        "achievements",
        col("achievements")
        .cast("double")
        .cast("integer"),
    )

    .withColumn(
        "dlc_count",
        col("dlc_count").cast("integer"),
    )

    # -------------------------------------------------
    # Языки
    #
    # Новый формат:
    # ["English", "French"]
    #
    # Старый формат:
    # English, French<strong>*</strong>, German
    # <br><strong>*</strong>languages with full audio support
    # -------------------------------------------------

    .withColumn(
        "_supported_languages_json",
        from_json(
            col("supported_languages"),
            string_array_schema,
        ),
    )

    .withColumn(
        "_full_audio_languages_json",
        from_json(
            col("full_audio_languages"),
            string_array_schema,
        ),
    )

    # У старого формата удаляем пояснение в конце строки.
    .withColumn(
        "_supported_languages_old_text",
        regexp_replace(
            col("supported_languages"),
            r"(?i)<br\s*/?>\s*<strong>\*</strong>\s*languages with full audio support.*$",
            "",
        ),
    )

    # Количество помеченных * языков = количество языков полной озвучки.
    .withColumn(
        "_old_full_audio_count",
        regexp_count(
            col("_supported_languages_old_text"),
            lit(r"<strong>\*</strong>"),
        ),
    )

    # Удаляем HTML-маркеры * перед подсчётом поддерживаемых языков.
    .withColumn(
        "_supported_languages_old_text",
        regexp_replace(
            col("_supported_languages_old_text"),
            r"<strong>\*</strong>",
            "",
        ),
    )

    .withColumn(
        "_supported_languages_old_array",
        array_filter(
            transform(
                split(
                    col("_supported_languages_old_text"),
                    ",",
                ),
                lambda item: trim(item),
            ),
            lambda item: length(item) > 0,
        ),
    )

    # На случай, если full_audio_languages встретится
    # в старом текстовом формате, а не JSON-массивом.
    .withColumn(
        "_full_audio_languages_old_array",
        array_filter(
            transform(
                split(
                    regexp_replace(
                        regexp_replace(
                            col("full_audio_languages"),
                            r"(?i)<br\s*/?>\s*<strong>\*</strong>\s*languages with full audio support.*$",
                            "",
                        ),
                        r"<strong>\*</strong>",
                        "",
                    ),
                    ",",
                ),
                lambda item: trim(item),
            ),
            lambda item: length(item) > 0,
        ),
    )

    # [] у supported_languages считаем отсутствием данных -> NULL.
    .withColumn(
        "supported_languages_count",
        when(
            col("_supported_languages_json").isNotNull(),
            when(
                size(col("_supported_languages_json")) == 0,
                lit(None).cast("integer"),
            ).otherwise(
                size(col("_supported_languages_json"))
            ),
        ).otherwise(
            when(
                col("supported_languages").isNull()
                | (trim(col("supported_languages")) == "")
                | (size(col("_supported_languages_old_array")) == 0),
                lit(None).cast("integer"),
            ).otherwise(
                size(col("_supported_languages_old_array"))
            )
        ),
    )

    # Для JSON [] означает отсутствие полной озвучки -> 0.
    # Для старого формата число берём по маркерам <strong>*</strong>.
    .withColumn(
        "full_audio_languages_count",
        when(
            col("_full_audio_languages_json").isNotNull(),
            size(col("_full_audio_languages_json")),
        ).when(
            col("full_audio_languages").isNotNull()
            & (trim(col("full_audio_languages")) != ""),
            size(col("_full_audio_languages_old_array")),
        ).when(
            col("_supported_languages_json").isNull()
            & col("supported_languages").isNotNull()
            & (trim(col("supported_languages")) != ""),
            col("_old_full_audio_count").cast("integer"),
        ).otherwise(
            lit(None).cast("integer")
        ),
    )

    # -------------------------------------------------
    # Поддержка ОС
    # -------------------------------------------------

    .withColumn(
        "windows",
        col("windows").cast("boolean"),
    )

    .withColumn(
        "mac",
        col("mac").cast("boolean"),
    )

    .withColumn(
        "linux",
        col("linux").cast("boolean"),
    )

    # -------------------------------------------------
    # Удаляем исходные и временные признаки
    # -------------------------------------------------

    .drop(
        "estimated_owners",
        "supported_languages",
        "full_audio_languages",
        "_estimated_owners_clean",
        "_supported_languages_json",
        "_full_audio_languages_json",
        "_supported_languages_old_text",
        "_supported_languages_old_array",
        "_full_audio_languages_old_array",
        "_old_full_audio_count",
    )
)


df.printSchema()


root
 |-- app_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- release_date: date (nullable = true)
 |-- price: double (nullable = true)
 |-- genres: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- positive: long (nullable = true)
 |-- negative: long (nullable = true)
 |-- recommendations: long (nullable = true)
 |-- peak_ccu: long (nullable = true)
 |-- metacritic_score: integer (nullable = true)
 |-- average_playtime_forever: long (nullable = true)
 |-- median_playtime_forever: long (nullable = true)
 |-- achievements: integer (nullable = true)
 |-- dlc_count: integer (nullable = true)
 |-- windows: boolean (nullable = true)
 |-- mac: boolean (nullable = true)
 |-- linux: boolean (nullable = true)
 |-- estimated_owners_min: long (nullable = true)
 |-- estimated_owners

## 8) Проверка результата нормализации

На этом этапе строки не удаляются и `NULL` не заполняются. Проверяем только, что разные исходные форматы были приведены к единой схеме.


In [8]:
from pyspark.sql.functions import sum as spark_sum

raw_count = df_raw.count()
processed_count = df.count()

print("Строк в исходном DataFrame:", f"{raw_count:,}")
print("Строк после преобразований:", f"{processed_count:,}")
print("Количество столбцов после преобразований:", len(df.columns))

if raw_count == processed_count:
    print("Количество строк не изменилось.")
else:
    print("ВНИМАНИЕ: количество строк изменилось.")

df.select(
    "app_id",
    "name",
    "estimated_owners_min",
    "estimated_owners_max",
    "estimated_owners_avg",
    "genres",
    "categories",
    "tags",
    "supported_languages_count",
    "full_audio_languages_count",
).show(20, truncate=False)

CHECK_COLUMNS = [
    "app_id",
    "name",
    "release_date",
    "price",
    "genres",
    "categories",
    "tags",
    "estimated_owners_min",
    "estimated_owners_max",
    "estimated_owners_avg",
    "supported_languages_count",
    "full_audio_languages_count",
]

df.select(
    [
        spark_sum(
            when(col(column).isNull(), 1).otherwise(0)
        ).alias(column)
        for column in CHECK_COLUMNS
    ]
).show(truncate=False)


Строк в исходном DataFrame: 140,243
Строк после преобразований: 140,243
Количество столбцов после преобразований: 24
Количество строк не изменилось.
+------+------------------------------+--------------------+--------------------+--------------------+-------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------+--------------------------+
|app_id|name                          |estimated_owners_min|estimated_owners_max|estimated_owners_avg|genres                                           |categories                                                                                                            

+------+----+------------+-----+------+----------+----+--------------------+--------------------+--------------------+-------------------------+--------------------------+
|app_id|name|release_date|price|genres|categories|tags|estimated_owners_min|estimated_owners_max|estimated_owners_avg|supported_languages_count|full_audio_languages_count|
+------+----+------------+-----+------+----------+----+--------------------+--------------------+--------------------+-------------------------+--------------------------+
|0     |16  |83          |1195 |0     |0         |0   |0                   |0                   |0                   |8397                     |19                        |
+------+----+------------+-----+------+----------+----+--------------------+--------------------+--------------------+-------------------------+--------------------------+



## 9) Сохранение в Iceberg


In [9]:
spark.sql(
    "CREATE NAMESPACE IF NOT EXISTS local.lab1"
)

(
    df.writeTo(TABLE_NAME)
    .using("iceberg")
    .createOrReplace()
)

print("Таблица успешно сохранена:", TABLE_NAME)

26/09/12 18:09:28 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Таблица успешно сохранена: local.lab1.steam_games


## 10) Проверка сохранённой таблицы


In [10]:
df_saved = spark.table(TABLE_NAME)

print("Количество строк:", f"{df_saved.count():,}")
print("Количество столбцов:", len(df_saved.columns))

df_saved.printSchema()
df_saved.show(10, truncate=False)

Количество строк: 140,243
Количество столбцов: 24
root
 |-- app_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- release_date: date (nullable = true)
 |-- price: double (nullable = true)
 |-- genres: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- positive: long (nullable = true)
 |-- negative: long (nullable = true)
 |-- recommendations: long (nullable = true)
 |-- peak_ccu: long (nullable = true)
 |-- metacritic_score: integer (nullable = true)
 |-- average_playtime_forever: long (nullable = true)
 |-- median_playtime_forever: long (nullable = true)
 |-- achievements: integer (nullable = true)
 |-- dlc_count: integer (nullable = true)
 |-- windows: boolean (nullable = true)
 |-- mac: boolean (nullable = true)
 |-- linux: boolean (nullable = true)
 |-- estimated_owners

## 11) Проверка каталога Iceberg


In [11]:
spark.sql(
    "SHOW TABLES IN local.lab1"
).show(truncate=False)


+---------+-----------+-----------+
|namespace|tableName  |isTemporary|
+---------+-----------+-----------+
|lab1     |steam_games|false      |
|lab1     |test_cars  |false      |
+---------+-----------+-----------+



## 12) Просмотр данных в таблице Iceberg


In [ ]:
spark.table("local.lab1.steam_games").show(10, truncate=False)